In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from scipy.stats import wilcoxon

### Defined the fold values

In [ ]:
# === Hardcoded 5-fold values (example values filled, update with yours) ===
fold_data = {
    "capreomycin": {
        "SD-LogReg": [
    0.8377652050919377,
    0.8194011117088041,
    0.8165612481551761,
    0.8031155081142063,
    0.8230704269955489
],
        "SD-CNN": [
        0.875, 
 0.8093672648514851, 
 0.8581872401951924, 
 0.84375, 
 0.8472685833243603
    ],
        "MD-CNN": [
        0.9464158001870495,
        0.8892895299145299,
        0.930271718714783,
        0.9061749432002597,
        0.9573335751859242,
    ],
        "SD-DNABERT-CNN-768": [0.8576379066478077, 0.8845436614667384, 0.8278726544381193, 0.866033151089126, 0.837743450799627],
        "SD-DNABERT-CNN-PCA10": [0.8540487977369164, 0.8753093060785367, 0.8121969217794643, 0.8478260869565217, 0.8019229754218056],
        "MD-DNABERT-CNN": [0.8488, 0.8152, 0.8415, 0.8287, 0.7854],
    },
    "amikacin": {
        "SD-LogReg": [
    0.9602803738317757,
    0.9822241902834008,
    0.9416147082334133,
    0.9568209568209568,
    0.9542282430213465
],
        "SD-CNN": [
        0.8707546458592484,
    0.8782427149964463,
    0.8506823083215587,
    0.8658913934426231,
    0.8291777814070808
    ],
        "MD-CNN": [
        0.9633297258297259,
        0.9889544711014176,
        0.9531456566611126,
        0.9592012580872768,
        0.9609574000878348,
    ],
        "SD-DNABERT-CNN-768": [0.817348, 0.924704, 0.879761, 0.830598, 0.864356],
        "SD-DNABERT-CNN-PCA10": [0.8149594358935478, 0.8864653447334909, 0.8398546964193047, 0.8189959432048681, 0.7687210190630164],
        "MD-DNABERT-CNN": [0.5000, 0.6400, 0.6667, 0.4821, 0.4939],
    },
    "kanamycin": {
        "SD-LogReg": [
    0.88230224296315,
    0.8483246844769043,
    0.846586129753915,
    0.8475814783507091,
    0.830272861356932
],
        "SD-CNN": [
        0.8770835957509342,
    0.8570707070707071,
    0.8601721938775511,
    0.8728363140127846,
    0.8665585275423731
    ],
        "MD-CNN": [
        0.9639322916666666,
        0.9715577318122504,
        0.9506055497925312,
        0.9544968181991388,
        0.9899774164408311,
    ],
        "SD-DNABERT-CNN-768": [0.875617498055538, 0.8350524270186004, 0.897503355704698, 0.875918098995022, 0.8357577433628319],
        "SD-DNABERT-CNN-PCA10": [0.8605242689874083, 0.8359493454630875, 0.8572527964205817, 0.8550108011646472, 0.8186301622418879],
        "MD-DNABERT-CNN": [0.8618, 0.8473, 0.8848, 0.8664, 0.9072],
    },
    "ethionamide": {
        "SD-LogReg": [
    0.5956850268443031,
 0.6774180444408335,
 0.6436673324038136,
 0.6008666826423835,
 0.5904749516872304
],
        "SD-CNN": [
        0.8884807824769252,
        0.8059610051264813,
        0.8720234375,
        0.88798928001161271,
        0.8693119266055047,
    ],
        "MD-CNN": [
        0.8904807824769252,
        0.8109610051264813,
        0.8740234375,
        0.8878928001161271,
        0.8793119266055047,
    ],
        "SD-DNABERT-CNN-768": [0.6508641975308641, 0.6055866965620329, 0.6129145532579009, 0.613656220322887, 0.6250154061049259],
        "SD-DNABERT-CNN-PCA10": [0.6271810699588477, 0.5825112107623318, 0.5800624268435428, 0.6097435897435898, 0.6363337578571135],
        "MD-DNABERT-CNN": [0.9103, 0.9127, 0.9100, 0.9243, 0.9007],
    },
    "pyrazinamide": {
        "SD-LogReg": [
    0.8671229000884174,
    0.8958427633348421,
    0.8817924764602744,
    0.8798051804517322,
    0.869690160141011
],
        "SD-CNN": [
        0.9165130572412981,
 0.9167575249571442,
 0.8920362500454033,
 0.9054563158360626,
 0.9336685222689466
    ],
        "MD-CNN": [
        0.9286268556005398,
        0.957183714944599,
        0.9414213679255906,
        0.9257427778023772,
        0.9579066659065139,
    ],
        "SD-DNABERT-CNN-768": [0.8840442086648983, 0.8933885029421605, 0.8324110888071206, 0.8752453245987728, 0.881874075868579],
        "SD-DNABERT-CNN-PCA10": [0.8820141467727673, 0.8913959026233443, 0.8573090451108987, 0.8741442944029152, 0.9033364799051017],
        "MD-DNABERT-CNN": [0.7065, 0.6274, 0.6782, 0.6930, 0.2882],
    },
    "moxifloxacin": {
        "SD-LogReg": [
    0.8435525826830175,
    0.8293851210517877,
    0.7923681257014591,
    0.8408050847457629,
    0.8201625799573561
],
        "SD-CNN": [
        0.8405400155400156,
    0.846891996891997,
    0.7889477140277436,
    0.8268779342723005,
    0.7915845301792922
    ],
        "MD-CNN": [
        0.9394735202492213,
        0.8884955434157561,
        0.9586907082521117,
        0.9450399665950847,
        0.9032253057676787,
    ],
        "SD-DNABERT-CNN-768": [0.7838907469342252, 0.7720458553791888, 0.7602813852813853, 0.8416313559322034, 0.861340618336887],
        "SD-DNABERT-CNN-PCA10": [0.7987365291713119, 0.7931297097963766, 0.7888608305274972, 0.8501483050847457, 0.8270477967306326],
        "MD-DNABERT-CNN": [0.9220, 0.9105, 0.9230, 0.9044, 0.9221],
    },
    "streptomycin": {
        "SD-LogReg": [
    0.9111163153786104,
    0.9176464456952261,
    0.9135333615668366,
    0.9223328476606767,
    0.912484590729783
],
        "SD-CNN": [
        0.9074986551909627,
 0.8981593250136185,
 0.912550495776717,
 0.9142405334837026,
 0.9310758547139513
    ],
        "MD-CNN": [
        0.9527992371923657,
        0.9572957260800849,
        0.9473854439368273,
        0.943516585294857,
        0.9499205030947598,
    ],
        "SD-DNABERT-CNN-768": [0.9059843871975019, 0.907410881801126, 0.8988918349935862, 0.8325908145602172, 0.804240631163708],
        "SD-DNABERT-CNN-PCA10": [0.8988836846213896, 0.8972378569939548, 0.8891850337881383, 0.9006192326372026, 0.8911335059171598],
        "MD-DNABERT-CNN": [0.9184, 0.9164, 0.9142, 0.9058, 0.9313],
    },
    "ethambutol": {
        "SD-LogReg": [0.9226361776276937,
    0.9132700205338807,
    0.9272763676950473,
    0.9290021395525984,
    0.9292855656262792],
        
        "SD-CNN": [
        0.9313150301991737,
    0.9234426485922835,
    0.9252429010720795,
    0.925338332984373,
    0.9237692355464471
    ],
        "MD-CNN": [
        0.9688726862158896,
        0.9491762894534258,
        0.9588726862158896,
        0.9543435692033823,
        0.9476301251331204,
    ],
        "SD-DNABERT-CNN-768": [0.9120345340974307, 0.9109779806894143, 0.9188392647108246, 0.910309188754028],
        "SD-DNABERT-CNN-PCA10": [0.9032502557619211, 0.9039571409847525, 0.922387938893575, 0.9086074985157555, 0.8945753028650104],
        "MD-DNABERT-CNN": [0.8969, 0.8922, 0.9111, 0.9216, 0.9217],
    },
    "levofloxacin": {
        "SD-LogReg": [
    0.9107142857142858,
    0.9839285714285715,
    0.901111111111111,
    0.9516129032258065,
    0.9107142857142857
],
        "SD-CNN": [
        0.8560606060606061,
    0.8757575757575757,
    0.7833333333333333,
    0.7921874999999999,
    0.9447115384615384
    ],
        "MD-CNN": [
        0.9285714285714286,
        0.9536679536679538,
        0.9571428571428571,
        0.9561128526645768,
        0.947008547008547,
    ],
        "SD-DNABERT-CNN-768": [0.9047619047619048, 0.9071428571428571, 0.9155555555555556, 0.5887096774193549, 0.8089285714285714],
        "SD-DNABERT-CNN-PCA10": [0.8738095238095238, 0.9321428571428572, 0.9199999999999999, 0.8373655913978495, 0.9196428571428571],
        "MD-DNABERT-CNN": [0.8617, 0.8355, 0.8821, 0.8640, 0.8928],
    },
    "isoniazid": {
        "SD-LogReg": [
    0.9244159436124488,
    0.9329562193126023,
    0.9357731137088203,
    0.9232263506663099,
    0.9274849899181392
],
        "SD-CNN": [
        0.8956103684893789,
    0.903917908309784,
    0.9187736445177543,
    0.9201996101499824,
    0.9229448160824671
    ],
        "MD-CNN": [
        0.9720486760201236,
        0.9794426332991408,
        0.968815552505883,
        0.9679437518711824,
        0.963787584431766,
    ],
        "SD-DNABERT-CNN-768": [0.9110567600916623, 0.9003988496496266, 0.8972390897390897, 0.8966296081419245, 0.8807091402104088],
        "SD-DNABERT-CNN-PCA10": [0.9099438862447853, 0.9008722244351334, 0.8938004563004563, 0.8945151402888258, 0.8771526743319987],
        "MD-DNABERT-CNN": [0.8009, 0.8126, 0.8363, 0.8381, 0.8259],
    },
    "rifampicin": {
        "SD-LogReg": [
    0.9685527394778496,
    0.9604744570801006,
    0.9595104386275863,
    0.964925451329179,
    0.9712580567843726
],
        "SD-CNN": [
        0.9720329460888977,
    0.9742235809073855,
    0.9707252352204065,
    0.9732912136350279,
    0.9717863249780648
    ],
        "MD-CNN": [
        0.9882049198465889,
        0.9782049198465889,
        0.9758334572358711,
        0.9836074881080251,
        0.9749294586257718,
    ],
        "SD-DNABERT-CNN-768": [0.9688105091188791, 0.9778577628309268, 0.9691001697792869, 0.9696248836621612, 0.971415024046603],
        "SD-DNABERT-CNN-PCA10": [0.9694322075379345, 0.9745219242974053, 0.9675808628270428, 0.9674417386266279, 0.9645045566098197],
        "MD-DNABERT-CNN": [0.8130, 0.7882, 0.8153, 0.8420, 0.8426],
    },
}

### Signifiance testing for SD cases with SD-LogReg as the baseline

In [ ]:
# === define models to compare ===
baseline = "SD-LogReg"
comparisons = [
    "SD-CNN",
    "SD-DNABERT-CNN",
    "SD-DNABERT-CNN-768",
    "SD-DNABERT-CNN-PCA10",
]

# === collect results ===
records = []

for drug, model_dict in fold_data.items():
    if baseline not in model_dict:
        continue

    base_vals = np.array(model_dict[baseline])

    for model in comparisons:
        if model not in model_dict:
            continue

        test_vals = np.array(model_dict[model])

        # Paired folds must match
        if len(base_vals) != len(test_vals) or len(base_vals) < 2:
            continue

        # --- Paired Wilcoxon signed-rank test (two-sided) ---
        w_stat, p_w = wilcoxon(
            test_vals,
            base_vals,
            alternative="two-sided"
        )

        records.append({
            "Drug": drug,
            "Model_vs_Baseline": model,
            "Mean_Baseline": np.mean(base_vals),
            "Mean_Model": np.mean(test_vals),
            "ΔAUC": np.mean(test_vals) - np.mean(base_vals),
            "W_stat": w_stat,
            "p_wilcoxon": p_w,
        })

# === create dataframe ===
res_df = pd.DataFrame(records)

# === sort results ===
res_df = res_df.sort_values(["Drug", "Model_vs_Baseline"])

# === display results ===
pd.set_option("display.max_rows", None)
print("\n Per-drug paired Wilcoxon tests against SD-LogReg:")
print(
    res_df[
        [
            "Drug", "Model_vs_Baseline",
            "Mean_Baseline", "Mean_Model", "ΔAUC",
            "W_stat", "p_wilcoxon"
        ]
    ].to_string(index=False, float_format="%.4f")
)

# === optional: summary across all drugs ===
print("\n Across all drugs summary (mean ΔAUC):")
print(res_df.groupby("Model_vs_Baseline")["ΔAUC"].mean())


### Signifiance testing for SD cases with SD-CNN as the baseline

In [ ]:
# === define models to compare ===
baseline = "SD-CNN"
comparisons = [
    "SD-DNABERT-CNN",
    "SD-DNABERT-CNN-768",
    "SD-DNABERT-CNN-PCA10",
]

# === collect results ===
records = []

for drug, model_dict in fold_data.items():
    if baseline not in model_dict:
        continue

    base_vals = np.array(model_dict[baseline])

    for model in comparisons:
        if model not in model_dict:
            continue

        test_vals = np.array(model_dict[model])

        # Paired folds must match
        if len(base_vals) != len(test_vals) or len(base_vals) < 2:
            continue

        # --- Paired Wilcoxon signed-rank test (two-sided) ---
        w_stat, p_w = wilcoxon(
            test_vals,
            base_vals,
            alternative="two-sided"
        )

        records.append({
            "Drug": drug,
            "Model_vs_Baseline": model,
            "Mean_Baseline": np.mean(base_vals),
            "Mean_Model": np.mean(test_vals),
            "ΔAUC": np.mean(test_vals) - np.mean(base_vals),
            "W_stat": w_stat,
            "p_wilcoxon": p_w,
        })

# === create dataframe ===
res_df = pd.DataFrame(records)

# === sort results ===
res_df = res_df.sort_values(["Drug", "Model_vs_Baseline"])

# === display results ===
pd.set_option("display.max_rows", None)
print("\n Per-drug paired Wilcoxon tests against SD-LogReg:")
print(
    res_df[
        [
            "Drug", "Model_vs_Baseline",
            "Mean_Baseline", "Mean_Model", "ΔAUC",
            "W_stat", "p_wilcoxon"
        ]
    ].to_string(index=False, float_format="%.4f")
)

# === optional: summary across all drugs ===
print("\n Across all drugs summary (mean ΔAUC):")
print(res_df.groupby("Model_vs_Baseline")["ΔAUC"].mean())


### Signifiance testing for MD cases with MD-CNN as the baseline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon

# === define drug groups ===
FIRST_LINE = ["isoniazid", "rifampicin", "ethambutol", "pyrazinamide"]
SECOND_LINE = ["streptomycin", "amikacin", "capreomycin", 
               "moxifloxacin", "levofloxacin", "ethionamide"]

def assign_group(drug):
    if drug in FIRST_LINE: return "First line"
    elif drug in SECOND_LINE: return "Second line"
    else: return "Other"

# === define models to compare ===
baseline = "MD-CNN"
comparisons = ["MD-DNABERT-CNN"]

# === collect per-fold AUCs into a dataframe ===
rows = []

for drug, model_dict in fold_data.items():
    for model, aucs in model_dict.items():
        for fold_idx, auc in enumerate(aucs):
            rows.append({
                "drug": drug,
                "model": model,
                "fold": fold_idx,
                "auc": auc,
                "line_group": assign_group(drug)
            })

df = pd.DataFrame(rows)

# === paired Wilcoxon tests ===
records = []

for drug, model_dict in fold_data.items():
    if baseline not in model_dict:
        continue

    base_vals = np.array(model_dict[baseline])

    for model in comparisons:
        if model not in model_dict:
            continue

        test_vals = np.array(model_dict[model])

        # Paired folds must match
        if len(base_vals) != len(test_vals) or len(base_vals) < 2:
            continue

        w_stat, p_w = wilcoxon(test_vals, base_vals, alternative="two-sided")

        records.append({
            "Drug": drug,
            "Model_vs_Baseline": model,
            "Mean_Baseline": np.mean(base_vals),
            "Mean_Model": np.mean(test_vals),
            "ΔAUC": np.mean(test_vals) - np.mean(base_vals),
            "W_stat": w_stat,
            "p_wilcoxon": p_w,
            "is_significant": p_w < 0.05
        })

res_df = pd.DataFrame(records).sort_values(["Drug", "Model_vs_Baseline"])

# === print Wilcoxon results ===
print("\n Per-drug paired Wilcoxon tests (MD-DNABERT-CNN vs MD-CNN):\n")
print(
    res_df[
        ["Drug", "Model_vs_Baseline", "Mean_Baseline", "Mean_Model", 
         "ΔAUC", "W_stat", "p_wilcoxon", "is_significant"]
    ].to_string(index=False, float_format="%.4f")
)

# === Panel A: pooled mean AUC ± 95% CI ===
summary = (
    df.groupby(["line_group","model"])
      .agg(mean_auc=("auc","mean"),
           std=("auc","std"),
           n=("auc","count"))
      .reset_index()
)

# classic 95% CI using 1.96 (your preferred formula)
summary["ci95_low"]  = summary["mean_auc"] - 1.96 * summary["std"] / np.sqrt(summary["n"])
summary["ci95_high"] = summary["mean_auc"] + 1.96 * summary["std"] / np.sqrt(summary["n"])

colors = {
    "MD-CNN": "teal",
    "MD-DNABERT-CNN": "blue",
}

fig, ax = plt.subplots(figsize=(7,5))

y_positions, labels = [], []
y = 0

for group in ["First line", "Second line"]:
    sub = summary[summary["line_group"] == group]
    for _, row in sub.iterrows():
        ax.errorbar(row["mean_auc"], y,
                    xerr=[[row["mean_auc"] - row["ci95_low"]],
                          [row["ci95_high"] - row["mean_auc"]]],
                    fmt="o",
                    color=colors.get(row["model"], "black"),
                    label=row["model"] if row["model"] not in labels else "")
        labels.append(row["model"])
        y_positions.append(y)
        y += 1
    y += 1  # spacing

ax.set_yticks(y_positions)
ax.set_yticklabels(labels)

ax.set_xlabel("Mean AUC")
ax.set_xlim(0.70, 1.00)
ax.set_title("a) Pooled mean AUC ± 95% CI")
ax.legend(title="Models")
plt.tight_layout()
plt.show()

# === Panel B: per-drug scatter of fold-level AUCs ===
fig, ax = plt.subplots(figsize=(10,5))
sns.stripplot(
    data=df,
    x="drug",
    y="auc",
    hue="model",
    dodge=True,
    jitter=False,
    alpha=0.7,
    palette=colors,
    ax=ax
)

ax.set_ylim(0.50, 1.0)
ax.set_title("b) Individual AUC values (5-fold CV)")
ax.set_ylabel("AUC")
ax.set_xlabel("Drug")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Models", bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.show()



 Per-drug paired Wilcoxon tests against SD-LogReg:
        Drug Model_vs_Baseline  Mean_Baseline  Mean_Model    ΔAUC  W_stat  p_wilcoxon
    amikacin    MD-DNABERT-CNN         0.9651      0.5565 -0.4086  0.0000      0.0625
 capreomycin    MD-DNABERT-CNN         0.9259      0.8239 -0.1020  0.0000      0.0625
  ethambutol    MD-DNABERT-CNN         0.9558      0.9087 -0.0471  0.0000      0.0625
 ethionamide    MD-DNABERT-CNN         0.8685      0.9116  0.0431  0.0000      0.0625
   isoniazid    MD-DNABERT-CNN         0.9704      0.8228 -0.1476  0.0000      0.0625
   kanamycin    MD-DNABERT-CNN         0.9661      0.8735 -0.0926  0.0000      0.0625
levofloxacin    MD-DNABERT-CNN         0.9485      0.8672 -0.0813  0.0000      0.0625
moxifloxacin    MD-DNABERT-CNN         0.9270      0.9164 -0.0106  5.0000      0.6250
pyrazinamide    MD-DNABERT-CNN         0.9422      0.5987 -0.3435  0.0000      0.0625
  rifampicin    MD-DNABERT-CNN         0.9802      0.8202 -0.1599  0.0000      0.0625
st

### CI tests

In [17]:
import pandas as pd
import numpy as np

# === collect all fold-level AUCs into a dataframe ===
rows = []
for drug, model_dict in fold_data.items():
    for model, aucs in model_dict.items():
        for fold_idx, auc in enumerate(aucs):
            rows.append({
                "drug": drug,
                "model": model,
                "fold": fold_idx,
                "auc": auc
            })

df = pd.DataFrame(rows)

# === compute per-drug summary statistics ===
summary = (
    df.groupby(["drug", "model"])
      .agg(
          mean_auc=("auc", "mean"),
          std=("auc", "std"),
          n=("auc", "count")
      )
      .reset_index()
)

# === 95% CI ===
summary["ci95_low"]  = summary["mean_auc"] - 1.96 * summary["std"] / np.sqrt(summary["n"])
summary["ci95_high"] = summary["mean_auc"] + 1.96 * summary["std"] / np.sqrt(summary["n"])

# === Add mean ± sd column (optional pretty format) ===
summary["mean_sd"] = summary.apply(
    lambda r: f"{r['mean_auc']:.4f} ± {r['std']:.4f}", axis=1
)

summary


,drug,model,mean_auc,std,n,ci95_low,ci95_high,mean_sd
0,amikacin,MD-CNN,0.965118,0.013848,5,0.952980,0.977256,0.9651 ± 0.0138
1,amikacin,MD-DNABERT-CNN,0.556540,0.089110,5,0.478431,0.634649,0.5565 ± 0.0891
2,amikacin,SD-CNN,0.858950,0.019458,5,0.841894,0.876006,0.8589 ± 0.0195
3,amikacin,SD-DNABERT-CNN-768,0.863353,0.042496,5,0.826104,0.900602,0.8634 ± 0.0425
4,amikacin,SD-DNABERT-CNN-PCA10,0.825799,0.042719,5,0.788354,0.863244,0.8258 ± 0.0427
5,amikacin,SD-LogReg,0.959034,0.014754,5,0.946101,0.971966,0.9590 ± 0.0148
6,capreomycin,MD-CNN,0.925897,0.028096,5,0.901270,0.950524,0.9259 ± 0.0281
7,capreomycin,MD-DNABERT-CNN,0.823920,0.025054,5,0.801959,0.845881,0.8239 ± 0.0251
8,capreomycin,SD-CNN,0.846715,0.024164,5,0.825534,0.867895,0.8467 ± 0.0242
9,capreomycin,SD-DNABERT-CNN-768,0.854766,0.022555,5,0.834996,0.874537,0.8548 ± 0.0226
